In [2]:
import jupedsim as jps 
from shapely import Polygon
import pathlib
import numpy as np
trajectory_file = pathlib.Path("simple_evac.sqlite")
if trajectory_file.exists():
    trajectory_file.unlink()
room = Polygon([(0, 0), (10, 0), (10, 8), (0, 8)])


exit_right = Polygon([
    (9.5, 3),
    (10, 3),
    (10, 5),
    (9.5, 5)
])
exit_left = Polygon([
    (0, 3),
    (0.5, 3),
    (0.5, 5),
    (0, 5)
])
start_positions = [(x,y) for x in range(3,8) for y in range(3,8)]

def create_simulation(policy,seed=1,record=False):
    rng = np.random.default_rng(seed)
    
    if record:
        writer =jps.SqliteTrajectoryWriter(
            output_file=pathlib.Path("simple_evac.sqlite"),
            every_nth_frame=5)
    else:
        writer = None
    simulation = jps.Simulation(
        model=jps.CollisionFreeSpeedModel(),
        geometry=room,
        trajectory_writer= writer
        
    )
    exit_id_right = simulation.add_exit_stage(exit_right)
    exit_id_left = simulation.add_exit_stage(exit_left)


    journey_left = jps.JourneyDescription([exit_id_left])
    journey_right = jps.JourneyDescription([exit_id_right])
    journey_id_left = simulation.add_journey(journey_left)
    journey_id_right = simulation.add_journey(journey_right)
    left = (journey_id_left,exit_id_left)
    right = (journey_id_right,exit_id_right)

    for pos in start_positions:
        route = choose_route(pos,policy,left,right,rng)
        parameters = jps.CollisionFreeSpeedModelAgentParameters(position=pos,journey_id=route[0],stage_id=route[1])
        simulation.add_agent(parameters)
    return simulation

def create_simulation_reroute(seed=1,record=False):
    rng = np.random.default_rng(seed)
    
    if record:
        writer =jps.SqliteTrajectoryWriter(
            output_file=pathlib.Path("simple_evac_reroute.sqlite"),
            every_nth_frame=5)
    else:
        writer = None
    simulation = jps.Simulation(
        model=jps.CollisionFreeSpeedModel(),
        geometry=room,
        trajectory_writer= writer
        
    )
    exit_id_right = simulation.add_exit_stage(exit_right)
    exit_id_left = simulation.add_exit_stage(exit_left)


    journey_left = jps.JourneyDescription([exit_id_left])
    journey_right = jps.JourneyDescription([exit_id_right])
    journey_id_left = simulation.add_journey(journey_left)
    journey_id_right = simulation.add_journey(journey_right)
    left = (journey_id_left,exit_id_left)
    right = (journey_id_right,exit_id_right)

    route = choose_route((5,5),'left',left,right,rng)
    parameters = jps.CollisionFreeSpeedModelAgentParameters(position=(5,5),journey_id=route[0],stage_id=route[1])
    reroute_agent_id = simulation.add_agent(parameters)
    return simulation,reroute_agent_id,right

def choose_route(position,policy,left,right,rng):
    if policy == 'left':
        return left
    elif policy == 'right':
        return right
    elif policy == 'random':
        if rng.random()<0.5:
            return left
        else:
            return right
    elif policy == 'spatial':
        if position[0] >5:
            return right
        elif position[0]<5:
            return left
        else:
            if rng.random()<0.5:
                return left
            else:
                return right
                    
    


steps = 0
simulation = create_simulation('random',record=True)
try:
    while simulation.agent_count() >0:
        simulation.iterate()
        steps+=1
finally:
    simulation._writer.close()
print(f"steps {steps}, time: {steps*simulation.delta_time()}, agent count {simulation.agent_count()}")


steps 877, time: 8.77, agent count 0


In [9]:
episodeNum=1
for i in range(episodeNum):
    simulation,agent_id,route = create_simulation_reroute(seed=i,record=True)
    print(agent_id)
    steps = 0
    switched = False
    while simulation.agent_count() >0:
        simulation.iterate()
        if simulation.elapsed_time()>=2 and not switched:
            simulation.switch_agent_journey(agent_id,route[0],route[1])
            switched = True
        steps+=1
    simulation._writer.close()
print('done')
            

54
done


In [2]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file
trajectory_data, walkable_area = read_sqlite_file("integratedgym_evac.sqlite")

animation = animate(
    trajectory_data,
    walkable_area,
    every_nth_frame=2
)

animation.show()

In [ ]:
policies = ["left", 'right','random','spatial']
episodeNum=1
for policy in ['left']:
    episodeSteps = []
    for i in range(episodeNum):
        simulation = create_simulation(policy=policy,seed=i)
        steps = 0    
        while simulation.agent_count() >0:
            simulation.iterate()
            if simulation.elapsed_time() % 0.10 == 0:
                for agent in simulation.agents():
                    print(
                        simulation.elapsed_time(),
                        agent.id,
                        agent.position
                    )                
                steps+=1
        episodeSteps.append(steps*simulation.delta_time())
    print(f'Num of episodes: {episodeNum}| policy: {policy}| average time: {np.mean(episodeSteps)}|stDeviation {np.std(episodeSteps)} | min {np.min(episodeSteps)}| max {np.max(episodeSteps)}')
            

In [1]:
from jupedsim_evac_env import JuPedSimEvacEnv

env = JuPedSimEvacEnv(record=True)

obs, info = env.reset(seed=1)

print(obs)
print(info)

terminated = False
truncated = False

total_reward = 0

while not (terminated or truncated):

    obs, reward, terminated, truncated, info = env.step(2)

    total_reward += reward
env.close()
print("Reward:", total_reward)
print("Evacuation time:", info["elapsed_time"])

[0.4 0.6 0.  0.  0.  0. ]
{'elapsed_time': 0.0, 'remaining_agents': 25}
Reward: -30.0
Evacuation time: 30.0


In [20]:
from gymnasium.utils.env_checker import check_env

check_env(JuPedSimEvacEnv())